In [2]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import DetectionModel
import pandas as pd
import pandas as pd

dataset_route='../datasets/coco_subset_5k/data.yaml'
dataset = ''
yolo = 'low/homeobjects/yolo26m'
baseline='(baseline:0,UNIFORM)'
experiment='(baseline:full,UNIFORM)8'

# load results
df = pd.read_csv(f"../runs/detect/{yolo}{dataset}_{baseline}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
bas = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}

print(bas)

df = pd.read_csv(f"../runs/detect/{yolo}{dataset}_{experiment}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
exp = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}


print(exp)
improvement_pct = {k: ((exp[k] - bas[k]) / bas[k] * 100) if bas[k] != 0 else float('inf') for k in exp}
print(improvement_pct)

model_exp  = YOLO(f'../runs/detect/{yolo}{dataset}_{experiment}/weights/best.pt')

model_baseline = YOLO(f'../runs/detect/{yolo}{dataset}_{baseline}/weights/best.pt')


metrics_baseline = model_baseline.val(
    data=dataset_route,
    imgsz=640,
    device=1,
    name=f'./evals/{yolo}{dataset}_{baseline}',
    verbose=False
)

metrics_exp = model_exp.val(
    data=dataset_route,
    imgsz=640,
    device=1,
    name=f'./evals/{yolo}{dataset}_{experiment}',
    verbose=False
)


# Get class names and maps values
classes = list(metrics_exp.names.values() if hasattr(metrics_baseline, "names") else metrics_baseline.names)
baseline_maps = metrics_baseline.maps if hasattr(metrics_baseline, "maps") else metrics_baseline.results_dict['maps']
exp_maps = metrics_exp.maps if hasattr(metrics_exp, "maps") else metrics_exp.results_dict['maps']


# Build the DataFrame
comparative_df = pd.DataFrame({
    "Class": classes,
    "Baseline mAP50-95": baseline_maps,
    "Experiment mAP50-95": exp_maps,
    "Improvement (%)": ((exp_maps - baseline_maps) / baseline_maps * 100)
})

{'precision': np.float64(0.5265), 'recall': np.float64(0.44594), 'mAP50': np.float64(0.45211), 'mAP50-95': np.float64(0.28828)}
{'precision': np.float64(0.66007), 'recall': np.float64(0.49018), 'mAP50': np.float64(0.53902), 'mAP50-95': np.float64(0.36701)}
{'precision': np.float64(25.369420702754052), 'recall': np.float64(9.920617123379827), 'mAP50': np.float64(19.223197894317764), 'mAP50-95': np.float64(27.310253919800203)}
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:1 (NVIDIA GeForce RTX 4060 Ti, 15948MiB)
YOLO26m summary (fused): 132 layers, 20,358,704 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3919.1±1475.3 MB/s, size: 151.0 KB)
val: Scanning /home/josericardopenase/Escritorio/datasets/coco_subset_5k/labels/val.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 110.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 1/32 1.4s/it 0.5s<44.

Process Process-6:
Process Process-8:
Process Process-4:

KeyboardInterrupt

Process Process-3:
Process Process-2:
Process Process-7:
Process Process-1:
Process Process-5:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/

In [6]:
# Display improvement_pct as a formatted DataFrame
improvement_df = pd.DataFrame(list(improvement_pct.items()), columns=["Metric", "Improvement (%)"])
print(improvement_df.to_string(index=False))

comparative_df

   Metric  Improvement (%)
precision         2.357254
   recall        -2.628940
    mAP50        -2.291508
 mAP50-95        -1.940634


,Class,Baseline mAP50-95,Experiment mAP50-95,Improvement (%)
0,industrial_ship,0.499189,0.507088,1.582332
1,military_ship,0.707117,0.689788,-2.450737
2,cruise_ship,0.745556,0.720715,-3.331858
3,passenger_ship,0.202876,0.230053,13.395749
4,sailboat,0.412205,0.407705,-1.091749
5,tugboat,0.407816,0.464718,13.952725
6,jetski,0.585172,0.548511,-6.264939
7,human_powered_boat,0.413078,0.421960,2.150193
8,small_boat,0.099382,0.091597,-7.833876
9,pedal_boat,0.451962,0.457359,1.194315


In [8]:
dataset = 'KITTI'
yolo = 'yolo26x'
experiment='(5750:5000,Uniform)'

# load results
df = pd.read_csv(f"../runs/detect/{yolo}{dataset}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
baseline = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}
baseline

{'precision': np.float64(0.64756),
 'recall': np.float64(0.54747),
 'mAP50': np.float64(0.61553),
 'mAP50-95': np.float64(0.37757)}